In [1]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1784626868995_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.

In [6]:
# PySpark SQL Functions

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    isnan,
    initcap,
    desc,
    asc,
    avg,
    sum,
    min,
    max,
    round,
    trim,
    lower,
    upper,
    split,
    explode,
    year,
    month,
    quarter,
    weekofyear,
    dayofmonth,
    datediff,
    current_date,
    lit,
    regexp_replace,
    create_map
)

from itertools import chain

# Window Functions

from pyspark.sql.window import Window


# PySpark Data Types

from pyspark.sql.types import (
    StringType,
    IntegerType,
    LongType,
    DateType,
    DoubleType
)

In [4]:
bronze_song_charts = spark.read.parquet(
    "s3://group-1-dbda/bronze/charts_songs_daily.parquet"
)

In [5]:
print("Rows:", bronze_song_charts.count())
print("Columns:", len(bronze_song_charts.columns))

('Rows:', 42757018)
('Columns:', 18)

In [7]:
bronze_song_charts.printSchema()

root
 |-- date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- label: string (nullable = true)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: date (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: date (nullable = true)
 |-- release_date: date (nullable = true)
 |-- artist_uris: string (nullable = true)

In [8]:
bronze_song_charts.rdd.getNumPartitions()

7

In [9]:
bronze_song_charts.cache()

bronze_song_charts.count()

42757018

Silver Starts below

In [10]:
silver_song_charts = bronze_song_charts

In [11]:
# Remove Duplicate Chart Events
silver_song_charts = (
    silver_song_charts
    .dropDuplicates(["date","country","uri"])
)

print("Rows:", silver_song_charts.count())

('Rows:', 42756826)

In [12]:
null_summary = (
    silver_song_charts
    .select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in silver_song_charts.columns
    ])
)

null_summary.show()

+----+-------+----+---+------------+----------+-----+---------+-------------+-------------+-------+----------------+------------+---------+----------+----------+------------+-----------+
|date|country|rank|uri|artist_names|track_name|label|peak_rank|previous_rank|days_on_chart|streams|consecutive_days|entry_status|peak_date|entry_rank|entry_date|release_date|artist_uris|
+----+-------+----+---+------------+----------+-----+---------+-------------+-------------+-------+----------------+------------+---------+----------+----------+------------+-----------+
|   0|      0|   0|  0|       44711|     44711|42993|        0|            0|            0|      0|               0|           0|        0|         0|         0|      636248|          0|
+----+-------+----+---+------------+----------+-----+---------+-------------+-------------+-------+----------------+------------+---------+----------+----------+------------+-----------+

In [14]:
# Country column Conversion
from itertools import chain
from pyspark.sql.functions import create_map

market_lookup = {
    "ad": "Andorra",
    "ae": "United Arab Emirates",
    "ar": "Argentina",
    "at": "Austria",
    "au": "Australia",
    "be": "Belgium",
    "bg": "Bulgaria",
    "bo": "Bolivia",
    "br": "Brazil",
    "ca": "Canada",
    "ch": "Switzerland",
    "cl": "Chile",
    "co": "Colombia",
    "cr": "Costa Rica",
    "cz": "Czech Republic",
    "de": "Germany",
    "dk": "Denmark",
    "do": "Dominican Republic",
    "ec": "Ecuador",
    "ee": "Estonia",
    "eg": "Egypt",
    "es": "Spain",
    "fi": "Finland",
    "fr": "France",
    "gb": "United Kingdom",
    "global": "Global",
    "gr": "Greece",
    "gt": "Guatemala",
    "hk": "Hong Kong",
    "hn": "Honduras",
    "hu": "Hungary",
    "id": "Indonesia",
    "ie": "Ireland",
    "il": "Israel",
    "in": "India",
    "is": "Iceland",
    "it": "Italy",
    "jp": "Japan",
    "kr": "South Korea",
    "lt": "Lithuania",
    "lu": "Luxembourg",
    "lv": "Latvia",
    "ma": "Morocco",
    "mx": "Mexico",
    "my": "Malaysia",
    "ni": "Nicaragua",
    "nl": "Netherlands",
    "no": "Norway",
    "nz": "New Zealand",
    "pa": "Panama",
    "pe": "Peru",
    "ph": "Philippines",
    "pl": "Poland",
    "pt": "Portugal",
    "py": "Paraguay",
    "ro": "Romania",
    "sa": "Saudi Arabia",
    "se": "Sweden",
    "sg": "Singapore",
    "sk": "Slovakia",
    "sv": "El Salvador",
    "th": "Thailand",
    "tr": "Turkey",
    "tw": "Taiwan",
    "ua": "Ukraine",
    "us": "United States",
    "uy": "Uruguay",
    "vn": "Vietnam",
    "za": "South Africa",
    "by": "Belarus",
    "kz": "Kazakhstan",
    "ng": "Nigeria",
    "pk": "Pakistan",
    "ve": "Venezuela"
}

mapping_expr = create_map(
    [lit(x) for x in chain(*market_lookup.items())]
)

silver_song_charts = (
    silver_song_charts
        .withColumnRenamed("country", "market")
        .withColumn(
            "country_name",
            mapping_expr[col("market")]
        )
)

In [8]:
# Handle Null Values
silver_song_charts = (
    silver_song_charts
    .fillna({
        "artist_names": "Unknown Artist",
        "track_name": "Unknown Track",
        "label": "Independent/Unknown"
    })
)

Data Validation + Cleaning

In [9]:
# Cleaning spaces
silver_song_charts = (
    silver_song_charts
    .withColumn("artist_names", trim(col("artist_names")))
)

In [10]:
# Create Artist Mapping Table : This makes a new df that has no collaborative artists. This tackles our artist analysis problem
from pyspark.sql.functions import arrays_zip

silver_artist_mapping = (
    silver_song_charts
    .select(
        "uri",
        explode(
            arrays_zip(
                split(col("artist_uris"), "\\|"),
                split(col("artist_names"), "\\|")
            )
        ).alias("artist")
    )
    .select(
        col("uri"),
        trim(col("artist.0")).alias("artist_uri"),
        trim(col("artist.1")).alias("artist_name")
    )
    .dropDuplicates()
)

In [11]:
# Create Valid Release Date
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "valid_release_date",
        when(
            col("release_date") <= col("date"),
            col("release_date")
        )
    )
)

Time Intelligence Features

In [12]:
#Create Date Features
silver_song_charts = (
    silver_song_charts
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("week", weekofyear(col("date")))
)

In [13]:
# Create Song Age Feature
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "song_age_days",
        datediff(
            col("date"),
            col("valid_release_date")
        )
    )
)

In [14]:
# Create Song Age Category
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "song_age_category",
        when(
            col("song_age_days") <= 90,
            "New Release"
        )
        .when(
            col("song_age_days") <= 365,
            "Recent Hit"
        )
        .when(
            col("song_age_days") <= 1825,
            "Established"
        )
        .when(
            col("song_age_days") > 1825,
            "Evergreen"
        )
        .otherwise("Unknown")
    )
)

In [15]:
silver_song_charts.printSchema()

root
 |-- date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = false)
 |-- track_name: string (nullable = false)
 |-- label: string (nullable = false)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: date (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: date (nullable = true)
 |-- release_date: date (nullable = true)
 |-- artist_uris: string (nullable = true)
 |-- valid_release_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- song_age_days: integer (nullable = true)
 |-- song_age_category: s

In [16]:
# Create Rank Movement Feature : if prev rank>0 then else dont. As prev rank = -1 means new entry
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "rank_movement",
        when(
            col("previous_rank") > 0,
            col("previous_rank") - col("rank")
        )
    )
)

In [17]:
# Create Movement Category
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "movement_category",
        when(
            col("rank_movement").isNull(),
            "New Entry"
        )
        .when(
            col("rank_movement") >= 50,
            "Strong Gainer"
        )
        .when(
            col("rank_movement") > 0,
            "Gainer"
        )
        .when(
            col("rank_movement") == 0,
            "Stable"
        )
        .when(
            col("rank_movement") <= -50,
            "Strong Decliner"
        )
        .otherwise("Decliner")
    )
)

In [18]:
# Create Hit Category
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "hit_category",
        when(col("rank") <= 10, "Global Hit")
        .when(col("rank") <= 50, "Major Hit")
        .when(col("rank") <= 100, "Popular Track")
        .otherwise("Charting Track")
    )
)

In [19]:
# Chart Strength Score

# based on:

#Rank performance
#Streams
#Longevity
#Peak performance
#(rank score) + (stream score) + (longevity score)

silver_song_charts = (
    silver_song_charts
    .withColumn(
        "chart_strength_score",
        (
            ((201 - col("rank")) * 0.4) +
            ((201 - col("peak_rank")) * 0.3) +
            (col("days_on_chart") * 0.2) +
            (col("consecutive_days") * 0.1)
        )
    )
)

In [20]:
# Create Stream Tier
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "stream_tier",
        when(col("streams") >= 10000000, "Mega Hit")
        .when(col("streams") >= 1000000, "Super Hit")
        .when(col("streams") >= 100000, "High Performer")
        .otherwise("Regular")
    )
)

In [16]:
silver_song_charts.printSchema()

root
 |-- date: date (nullable = true)
 |-- market: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- label: string (nullable = true)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: date (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: date (nullable = true)
 |-- release_date: date (nullable = true)
 |-- artist_uris: string (nullable = true)
 |-- country_name: string (nullable = true)

In [27]:
len(silver_song_charts.columns)

31

Create Standardized Label & artists

In [23]:
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "standardized_label",
        initcap(
            lower(
                trim(col("label"))
            )
        )
    )
)

In [24]:
silver_artist_mapping = (
    silver_artist_mapping
    .withColumn(
        "standardized_artist_name",
        initcap(
            lower(
                trim(col("artist_name"))
            )
        )
    )
)

In [25]:
silver_artist_mapping.printSchema()

root
 |-- uri: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- standardized_artist_name: string (nullable = true)

Write silver_song_charts

In [28]:
silver_song_charts_optimized = (
    silver_song_charts
    .repartition(
        60,
        "year",
        "country"
    )
)

In [29]:
silver_song_charts_optimized.rdd.getNumPartitions()

60

In [30]:
(
    silver_song_charts_optimized
    .write
    .mode("overwrite")
    .partitionBy(
        "year",
        "country"
    )
    .parquet(
        "s3://group-1-dbda/silver/song_charts/"
    )
)

In [32]:
silver_artist_mapping_optimized = (
    silver_artist_mapping
    .repartition(
        40,
        "artist_uri"
    )
)

In [34]:
(
    silver_artist_mapping_optimized
    .write
    .mode("overwrite")
    .parquet(
        "s3://group-1-dbda/silver/artist_mapping/"
    )
)

Read Silver Song Charts from S3

In [35]:
silver_song_test = spark.read.parquet(
    "s3://group-1-dbda/silver/song_charts/"
)

In [15]:
silver_song_test.printSchema()

name 'silver_song_test' is not defined
Traceback (most recent call last):
NameError: name 'silver_song_test' is not defined



In [37]:
silver_song_test.count()

42756826

In [38]:
len(silver_song_charts.columns)

31

In [39]:
# Verify Partition Columns
silver_song_test.select(
    "year",
    "country"
).distinct().count()

672

In [40]:
# Sample Business Columns
silver_song_test.select(
    "track_name",
    "standardized_label",
    "song_age_category",
    "movement_category",
    "hit_category",
    "stream_tier",
    "chart_strength_score"
).show(10, False)

+-----------------------+------------------------------------+-----------------+-----------------+--------------+--------------+--------------------+
|track_name             |standardized_label                  |song_age_category|movement_category|hit_category  |stream_tier   |chart_strength_score|
+-----------------------+------------------------------------+-----------------+-----------------+--------------+--------------+--------------------+
|Lucid Dreams           |Juice Wrld Mixtape / Isr P&d        |Evergreen        |New Entry        |Charting Track|Regular       |323.90000000000003  |
|Believe                |Warner Records                      |Unknown          |Strong Decliner  |Charting Track|Regular       |47.900000000000006  |
|HDL                    |Lacazette                           |New Release      |New Entry        |Charting Track|Regular       |103.1               |
|Beanie                 |Chezile / 10k Projects              |Established      |Gainer           |Ch

In [41]:
silver_artist_test = spark.read.parquet(
    "s3://group-1-dbda/silver/artist_mapping/"
)

print(silver_artist_test.count())
print(len(silver_artist_test.columns))

silver_artist_test.show(10, False)

379930
4
+------------------------------------+-------------------------------------+-----------+------------------------+
|uri                                 |artist_uri                           |artist_name|standardized_artist_name|
+------------------------------------+-------------------------------------+-----------+------------------------+
|spotify:track:4wopYJ5wpYOM6ogm2Jugnw|spotify:artist:1zNqDE7qDGCsyzJwohVaoX|Anne-Marie |Anne-marie              |
|spotify:track:5AIoKEMANJ7iy2oDhPQaMw|spotify:artist:4xklx5DAtVru5uf3vSXTgf|Arman Aydin|Arman Aydin             |
|spotify:track:1kXHqvq1R8dkx1Mm52nH9g|spotify:artist:4WN5naL3ofxrVBgFpguzKo|Rudimental |Rudimental              |
|spotify:track:4SUjsrUM5F05F85ZdAZn2v|spotify:artist:6chmC6o0wvACYVGTITw3Pz|NONT TANONT|Nont Tanont             |
|spotify:track:6a9Z1jUms915w4O7N1PxjY|spotify:artist:1fctva4kpRbg2k3v7kwRuS|Rvssian    |Rvssian                 |
|spotify:track:3uI1I2juxVWkrjSHdd9mVh|spotify:artist:4yxLYO2imECxGYTTV7RQKb|Pai